In [136]:
import pandas as pd

google_trends = pd.read_csv("../data/gold_google_trends_daily.csv")
data = pd.read_csv("../files/processed_data.csv")

In [137]:
interest = data.drop(["away_team_code", "last_result_vs_opponent", "weekday", "date", "kickoff_time_local"], axis=1)

In [138]:
# google_trends["segment"] = google_trends["match_id"].notna().cumsum()
google_trends["segment"] = google_trends["match_id"].notna().shift(fill_value=False).cumsum()

In [139]:
google_trends["has_future_match"] = google_trends["match_id"].notna()[::-1].cummax()[::-1]
google_trends = google_trends[google_trends["has_future_match"]]

In [140]:
no_match_days = google_trends[google_trends["match_id"].isna()]

result = (
    no_match_days
    .groupby("segment")["ohl_interest"]
    .agg(["sum", "count"])
)

result["avg"] = result["sum"] / result["count"]

In [141]:
result

,sum,count,avg
segment,,,
0,92.37,22,4.198636
1,23.70,6,3.950000
2,40.41,7,5.772857
3,33.42,6,5.570000
4,26.44,6,4.406667
...,...,...,...
137,17.00,5,3.400000
138,53.00,6,8.833333
139,18.00,6,3.000000


In [142]:
result_2 = (
    google_trends
    .groupby("segment")["ohl_interest"]
    .agg(["sum", "count"])
)

result_2["avg"] = result_2["sum"] / result_2["count"]

In [143]:
avg = result_2.drop(["sum", "count"], axis=1)

In [144]:
ids = pd.DataFrame(google_trends.dropna()["match_id"].unique(), columns=["match_id"])
avg_data = pd.concat([avg, ids], axis=1)

# avg_data
# interest_data = pd.concat()
interest = pd.merge(interest, avg_data, on="match_id")

In [145]:
interest.drop("match_id", axis=1).corr()
# interest

,tickets_scanned,ohl_interest,avg
tickets_scanned,1.000000,0.081318,0.013364
ohl_interest,0.081318,1.000000,0.354749
avg,0.013364,0.354749,1.000000
